# GenAI 데이터 카운트 확인

각 카테고리의 `data_augmented/{cat}/gen_ai/annotations.json`을 읽어서:
- 총 이미지 수
- 클래스별 annotation 수
- 클래스별 이미지 수 (해당 클래스가 등장한 이미지)

In [ ]:
import json
from pathlib import Path
from collections import Counter, defaultdict

PROJECT_ROOT = Path('/home/jjh0709/gitrepo/VISION-Instance-Seg')
CATEGORIES = ['Cable', 'Screw', 'Casting', 'Console', 'Cylinder', 'Wood']

In [ ]:
def check_genai(category):
    p = PROJECT_ROOT / 'data_augmented' / category / 'gen_ai' / 'annotations.json'
    if not p.exists():
        print(f'{category}: 파일 없음 ({p})')
        return None
    
    d = json.load(open(p))
    
    # 카테고리 매핑
    cat_map = {c['id']: c['name'] for c in d.get('categories', [])}
    
    # 클래스별 annotation 수
    ann_count = Counter(a['category_id'] for a in d['annotations'])
    
    # 클래스별 이미지 수 (한 이미지에 여러 클래스 있을 수 있음)
    img_classes = defaultdict(set)
    for a in d['annotations']:
        img_classes[a['image_id']].add(a['category_id'])
    
    img_per_cls = Counter()
    for img_id, cls_set in img_classes.items():
        for c in cls_set:
            img_per_cls[c] += 1
    
    return {
        'category': category,
        'file': str(p),
        'mtime': p.stat().st_mtime,
        'total_images': len(d['images']),
        'total_annotations': len(d['annotations']),
        'cat_map': cat_map,
        'ann_count': dict(ann_count),
        'img_per_cls': dict(img_per_cls),
    }

In [ ]:
# 모든 카테고리 출력
from datetime import datetime

for cat in CATEGORIES:
    info = check_genai(cat)
    if info is None:
        continue
    
    mtime = datetime.fromtimestamp(info['mtime']).strftime('%Y-%m-%d %H:%M')
    print(f"=== {cat} ===")
    print(f"  파일 수정일: {mtime}")
    print(f"  총 이미지: {info['total_images']}장")
    print(f"  총 annotations: {info['total_annotations']}건")
    print(f"  클래스별 annotation 수:")
    for cid, cnt in sorted(info['ann_count'].items()):
        name = info['cat_map'].get(cid, '?')
        print(f"    [{cid}] {name}: {cnt}건")
    print(f"  클래스별 이미지 수 (해당 클래스가 등장한 이미지):")
    for cid, cnt in sorted(info['img_per_cls'].items()):
        name = info['cat_map'].get(cid, '?')
        print(f"    [{cid}] {name}: {cnt}장")
    print()

In [ ]:
# 표로 정리
import pandas as pd

rows = []
for cat in CATEGORIES:
    info = check_genai(cat)
    if info is None:
        continue
    for cid in sorted(info['cat_map'].keys()):
        rows.append({
            'category': cat,
            'class_id': cid,
            'class_name': info['cat_map'][cid],
            'images': info['img_per_cls'].get(cid, 0),
            'annotations': info['ann_count'].get(cid, 0),
        })

df = pd.DataFrame(rows)
df

In [ ]:
# genai_125 조건 충족 여부 확인
TARGET = 125
print(f"클래스당 {TARGET}장 이상 확보 여부:\n")
for _, row in df.iterrows():
    status = 'OK' if row['annotations'] >= TARGET else 'FAIL'
    print(f"  [{status}] {row['category']:<10s} {row['class_name']:<15s}: {row['annotations']}건 (이미지 {row['images']}장)")